# Lesson 7

## Video 1 — Lesson Introduction and Model Family

In [19]:
# Load environment variables
from dotenv import load_dotenv
load_dotenv()

# Import libraries
import anthropic
import time

# Import our model helpers
from models import (
    HAIKU_MODEL,
    SONNET_MODEL,
    OPUS_MODEL,
    FABLE_MODEL,
    EFFORT_LEVELS,
    pick_model,
    read_usage,
)

# Create the client
client = anthropic.Anthropic()

### Claude Code vs API

- Claude Code in VS Code can use an eligible Claude subscription.
- Python SDK calls use the Messages API and API billing.

## Video 2 — Haiku, Sonnet, Opus, and Fable

### Haiku 5.5 — Fast Classification

In [20]:
# A simple high-volume task
haiku_prompt = '''
Classify each support ticket as:
billing, technical, account, or feature_request.

1. We were charged twice this month.
2. The dashboard is blank for all users.
3. Can we add SSO next quarter?
4. I cannot sign in after resetting my password.

Return only the ticket number and category.
'''

resp = client.messages.create(
    model=HAIKU_MODEL,
    max_tokens=200,
    messages=[
        {"role": "user", "content": haiku_prompt}
    ]
)

reply = "".join(block.text for block in resp.content if block.type == "text")
print(reply)

1. billing
2. technical
3. feature_request
4. account


### Sonnet 5.5 — Writing in a Given Tone

In [21]:
# A writing task that needs good style control
sonnet_prompt = '''
Study this writing style:

"Most AI demos are impressive for 5 minutes.
The real question is what happens on day 50."

"Start with the workflow.
Then decide whether you even need an agent."

Now write 2 short posts about choosing the right AI model.

Keep the same direct, technical tone.
No hashtags.
'''

resp = client.messages.create(
    model=SONNET_MODEL,
    max_tokens=300,
    messages=[
        {"role": "user", "content": sonnet_prompt}
    ]
)

reply = "".join(block.text for block in resp.content if block.type == "text")
print(reply)

**Post 1**

Most teams pick a model by leaderboard rank.
Then they discover it's too slow, too expensive, or fails on their actual data.

Build a test set from your real tasks first.
Run every candidate against it. The winner is rarely the one at the top of the benchmark.

**Post 2**

Don't ask which model is best.
Ask what a wrong answer costs you.

If the cost is low, use the cheap, fast model.
If the cost is high, pay for the stronger one, and add verification either way.


### Opus 5.5 — Harder Engineering Task

In [22]:
# A more difficult reasoning task
opus_prompt = '''
A Python asyncio service occasionally processes the same job twice.

Explain:
1. what kind of concurrency problem could cause this,
2. why it happens,
3. one safe way to fix it in a single-process asyncio application.

Keep the answer practical.
'''

resp = client.messages.create(
    model=OPUS_MODEL,
    max_tokens=500,
    messages=[
        {"role": "user", "content": opus_prompt}
    ]
)

reply = "".join(block.text for block in resp.content if block.type == "text")
print("Claude:", reply)

Claude: ## 1. The likely problem: a check-then-act race

The usual cause is a **race condition** where a worker *checks* whether a job is available and *claims* it in two separate steps, with an `await` between them:

```python
async def worker():
    while True:
        job = await get_next_pending_job()     # step 1: check
        # ... another coroutine can run here ...
        await mark_job_processing(job.id)      # step 2: claim
        await process(job)
```

## 2. Why it happens even though asyncio is single-threaded

asyncio doesn't run coroutines in parallel, but it **switches between them at every `await`**. Any `await` is a point where another task can run.

A typical interleaving:

| Worker A | Worker B |
|---|---|
| `get_next_pending_job()` → job 42 (pending) | |
| *awaits DB / network, yields control* | |
| | `get_next_pending_job()` → job 42 (still pending


### Fable 5.1 — Optional Advanced Example

In [23]:
# Optional prompt example
fable_prompt = '''
Review a company website as if you were advising the founder.

Focus on:
- positioning
- target customer
- trust signals
- conversion path
- the 5 highest-impact improvements

review this website page: https://www.anasriad.com/ai
'''

resp = client.messages.create(
    model=FABLE_MODEL,
    max_tokens=4000,
    tools=[{
        "type": "web_fetch_20260318",   # web fetch version for Fable 5 / 5.1
        "name": "web_fetch",
        "max_uses": 3,
        "allowed_domains": ["anasriad.com"],
    }],
    messages=[{"role": "user", "content": fable_prompt}],
)

reply = "".join(block.text for block in resp.content if block.type == "text")
print(reply)
print(resp.stop_reason, resp.usage)

I'll start by fetching the page so I can see what a visitor actually sees.I've read the full page (copy, structure, metadata, and links). One caveat up front: I'm working from the page's content and structure, not a rendered screenshot, so I can't comment on visual design, load speed or mobile layout. Everything below is about what the page *says* and how it moves a visitor toward a booking.

## Overall verdict

This is a far better page than most solo AI consultants ship. The positioning is specific, the offer is productised (£400 audit → fixed-price build → optional retainer), the anti-positioning is sharp, and there's a single consistent CTA. The problems are mostly about **proof not yet matching the ambition of the ICP**, a handful of **credibility leaks**, and a **conversion path that only has one door**.

---

## 1. Positioning

**What's working**
- The headline sells an outcome, not a technology: *"Grow your business without hiring more people for admin."* No "AI agents", no "RA

## Video 3 — Choosing and Routing Models

In [24]:
# Test the routing function
print("Simple:", pick_model("simple"))
print("Standard:", pick_model("standard"))
print("Hard:", pick_model("hard"))
print("Frontier:", pick_model("frontier"))

Simple: claude-haiku-5-5
Standard: claude-sonnet-5-5
Hard: claude-opus-5-5
Frontier: claude-fable-5-1


In [25]:
# Example task routing
task = "simple"
model = pick_model(task)

print("Task:", task)
print("Chosen model:", model)

Task: simple
Chosen model: claude-haiku-5-5


In [26]:
# See the available effort levels
print(EFFORT_LEVELS)

['low', 'medium', 'high', 'xhigh', 'max']


In [31]:
# Ask Sonnet to use high effort for a harder task
reasoning_prompt = '''
A product has 1,000 users and a 5% monthly churn rate.

Give 3 practical actions the company should test first to reduce churn,
and briefly explain why each one could help.
'''

resp = client.messages.create(
    model=SONNET_MODEL,
    max_tokens=400,
    output_config={"effort": "low"},
    messages=[
        {"role": "user", "content": reasoning_prompt}
    ]
)

reply = "".join(block.text for block in resp.content if block.type == "text")
print(reply)

With 1,000 users and 5% monthly churn, you're losing about 50 users a month, and roughly 46% of your base over a year. These are the three tests I'd run first.

**1. Improve onboarding to reach the "aha moment" faster**
- **Test:** Add a guided setup, checklist, or triggered emails/in-app prompts that push new users toward the key action that predicts retention, such as creating their first project or inviting a teammate. Compare against the current flow.
- **Why it helps:** Early-life churn is usually the biggest leak. Users who never see the core value leave quietly. Getting more people to that value in the first days often has the largest effect on long-term retention.

**2. Identify at-risk users and intervene proactively**
- **Test:** Define a simple risk signal, such as no login in 14 days or a sharp drop in usage. Then send a targeted outreach to a random half of those users, such as a helpful tip, a personal check-in, or a short offer of help, and leave the other half as a cont

In [33]:
# Try the same task with low effort
resp_low = client.messages.create(
    model=OPUS_MODEL,
    max_tokens=400,
    output_config={"effort": "xhigh"},
    messages=[
        {"role": "user", "content": reasoning_prompt}
    ]
)

reply = "".join(block.text for block in resp.content if block.type == "text")
print(reply)

With 1,000 users and 5% monthly churn, you're losing about 50 users a month, and roughly 46% of your base over a year. These are the three tests I'd run first.

**1. Improve onboarding to reach the "aha moment" faster**
- **Test:** Add a guided setup, checklist, or triggered emails/in-app prompts that push new users toward the key action that predicts retention, such as creating their first project or inviting a teammate. Compare against the current flow.
- **Why it helps:** Early-life churn is usually the biggest leak. Users who never see the core value leave quietly. Getting more people to that value in the first days often has the largest effect on long-term retention.

**2. Identify at-risk users and intervene proactively**
- **Test:** Define a simple risk signal, such as no login in 14 days or a sharp drop in usage. Then send a targeted outreach to a random half of those users, such as a helpful tip, a personal check-in, or a short offer of help, and leave the other half as a cont

## Video 4 — Reading Token Usage

In [34]:
# Make a request we can inspect
resp = client.messages.create(
    model=SONNET_MODEL,
    max_tokens=150,
    messages=[
        {
            "role": "user",
            "content": "Explain model routing in two short sentences."
        }
    ]
)

reply = "".join(block.text for block in resp.content if block.type == "text")
print("Claude:", reply)

Claude: Model routing is the practice of automatically directing each incoming request to the most suitable AI model based on factors like task complexity, cost, speed, or required capability. This lets systems send simple queries to small, cheap models and reserve larger, more powerful ones for harder tasks, balancing quality against cost and latency.


In [35]:
# Read the token usage
input_tokens, output_tokens = read_usage(resp)

print("Input tokens:", input_tokens)
print("Output tokens:", output_tokens)

Input tokens: 21
Output tokens: 100


In [36]:
# Simple live timing
start = time.time()

resp = client.messages.create(
    model=SONNET_MODEL,
    max_tokens=150,
    messages=[
        {
            "role": "user",
            "content": "Why can a smaller model be useful in production?"
        }
    ]
)

end = time.time()

reply = "".join(block.text for block in resp.content if block.type == "text")
print("Claude:", reply)
print("Time:", round(end - start, 2), "seconds")

Claude: A smaller model can be the better choice in production for several reasons.

**Cost**
- Less compute per request, so lower GPU/CPU bills at scale
- Can run on cheaper hardware, or more instances fit on the same hardware
- Lower memory footprint means higher batch sizes and better utilization

**Latency**
- Faster inference makes real-time use cases viable (autocomplete, voice, fraud checks, search ranking)
- Lower and more pred
Time: 2.08 seconds


## Video 5 — Comparing Models and Cost

In [37]:
comparison_prompt = '''
A SaaS company has rising churn.

Customers say:
- onboarding takes too long,
- support replies are slow,
- users do not understand one key feature.

Give 3 practical actions the company should test first.
'''

In [38]:
# Compare Haiku and Sonnet
for model in [HAIKU_MODEL, SONNET_MODEL]:
    start = time.time()

    resp = client.messages.create(
        model=model,
        max_tokens=250,
        messages=[
            {"role": "user", "content": comparison_prompt}
        ]
    )

    end = time.time()
    input_tokens, output_tokens = read_usage(resp)

    print("\nMODEL:", model)
    print("Time:", round(end - start, 2), "seconds")
    print("Input:", input_tokens)
    print("Output:", output_tokens)
    reply = "".join(block.text for block in resp.content if block.type == "text")
    print("Claude:", reply)


MODEL: claude-haiku-5-5
Time: 2.17 seconds
Input: 75
Output: 250
Claude: Start with three low-cost experiments that target the complaints most likely to drive churn. Each one should have a clear metric and a comparison group.

**1. Shorten onboarding to the first "aha" moment**
- **Action:** Identify

MODEL: claude-sonnet-5-5
Time: 3.15 seconds
Input: 75
Output: 250
Claude: # 3 Actions to Test First

## 1. Shorten onboarding with a guided "first win" path
**Test:** Build a simplified onboarding flow (checklist, templates, or pre-filled sample data) that gets new users to one meaningful outcome quickly. Offer it to a random 50% of new signups.
- **Measure:** Time-to-first-value, onboarding completion rate, 30/60-day retention
- **Why first:** Early experience strongly predicts long-term retention, and this addresses the first point where customers disengage.

## 2. Triage support and set response-time targets
**Test:** Create a priority queue so churn-risk accounts (new customers, high